In [9]:
import os
from pathlib import Path

import polars as pl
from sklearn.model_selection import train_test_split

In [16]:
DATASET_PATH = Path("datasets/dataset_20260707.csv")

In [17]:
DATABASE_URI = os.environ["DATABASE_URI"]

In [18]:
RANDOM_SEED = 42

# Chargement des données

In [19]:
df_dataset = pl.read_csv(DATASET_PATH, schema_overrides={"cluster_id":pl.String})

In [20]:
df_dataset

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true,"""7758461304653219539"""
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false,null
…,…,…,…
"""refashion_TLC-REFASHION-PAV-32…","""refashion_TLC-REFASHION-PAV-34…",true,"""9a14e7d6-f708-58fd-bcbb-3d3c2c…"
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true,"""1f8fb158-8b9a-54b5-81ca-d9d542…"
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true,"""05f37598-e1d8-40a1-bb9b-5be142…"


In [21]:
df_dataset.group_by("label").len()

label,len
bool,u32
true,1129
false,1000


# Chargement des variables d'intérêt depuis la base

In [22]:
df_dataset.write_database("luis.dataset_tmp", connection=DATABASE_URI,if_table_exists="replace")

129

In [23]:
sql_dataset_features = """
with actions_by_actor as (
-- D'abord on calcule les actions des parents sur la base de leurs enfants
select 
	qv.identifiant_unique as acteur_id,
	(1 = any(array_agg(qv2.action_id))) as action_reparer,
	(2 = any(array_agg(qv2.action_id))) as action_acheter,
	(3 = any(array_agg(qv2.action_id))) as action_revendre,
	(4 = any(array_agg(qv2.action_id))) as action_donner,
	(5 = any(array_agg(qv2.action_id))) as action_louer,
	(6 = any(array_agg(qv2.action_id))) as action_mettreenlocation,
	(7 = any(array_agg(qv2.action_id))) as action_emprunter,
	(8 = any(array_agg(qv2.action_id))) as action_preter,
	(9 = any(array_agg(qv2.action_id))) as action_echanger,
	(11 = any(array_agg(qv2.action_id))) as action_trier,
	(12 = any(array_agg(qv2.action_id))) as action_rapporter
from
	qfdmo_vueacteur qv
inner join qfdmo_vueacteur qv3 on
	qv.est_parent
	and qv.identifiant_unique = qv3.parent_id
	and qv3.statut <> 'SUPPRIME'
inner join qfdmo_vuepropositionservice qv2 on
	qv3.identifiant_unique = qv2.acteur_id
group by
	1
union all -- on joint les actions des enfants
select
	acteur_id,
	(1 = any(array_agg(action_id))) as action_reparer,
	(2 = any(array_agg(action_id))) as action_acheter,
	(3 = any(array_agg(action_id))) as action_revendre,
	(4 = any(array_agg(action_id))) as action_donner,
	(5 = any(array_agg(action_id))) as action_louer,
	(6 = any(array_agg(action_id))) as action_mettreenlocation,
	(7 = any(array_agg(action_id))) as action_emprunter,
	(8 = any(array_agg(action_id))) as action_preter,
	(9 = any(array_agg(action_id))) as action_echanger,
	(11 = any(array_agg(action_id))) as action_trier,
	(12 = any(array_agg(action_id))) as action_rapporter
from
	qfdmo_vuepropositionservice
group by
	acteur_id
),
-- Sélection des variables à la maille acteur avec les actions précédemment sélectionnées
features as (
select
	identifiant_unique,
	nom,
	description,
	acteur_type_id,
	adresse,
	adresse_complement,
	code_postal,
	ville,
	url,
	email,
	"location",
	telephone,
	nom_commercial,
	nom_officiel,
	siren,
	siret,
	source_id,
	naf_principal,
	horaires_osm,
	horaires_description,
	public_accueilli,
	reprise,
	exclusivite_de_reprisereparation,
	uniquement_sur_rdv,
	consignes_dacces,
	action_principale_id,
	lieu_prestation,
	latitude,
	longitude,
	code_commune_insee,
	epci_id,
	aba.*
from
	qfdmo_vueacteur qv
left join actions_by_actor aba on
	qv.identifiant_unique = aba.acteur_id)
-- Join avec les acteurs sélectionnés dans le dataset
select 
	dt.*,
	f.nom as nom_i,
	f.description as description_i,
	f.acteur_type_id as acteur_type_id_i,
	f.adresse as adresse_i,
	f.adresse_complement as adresse_complement_i,
	f.code_postal as code_postal_i,
	f.ville as ville_i,
	f.url as url_i,
	f.email as email_i,
	f.telephone as telephone_i,
	f.nom_commercial as nom_commercial_i,
	f.nom_officiel as nom_officiel_i,
	f.siren as siren_i,
	f.siret as siret_i,
	f.source_id as source_id_i,
	f.naf_principal as naf_principal_i,
	f.horaires_osm as horaires_osm_i,
	f.horaires_description as horaires_description_i,
	f.public_accueilli as public_accueilli_i,
	f.reprise as reprise_i,
	f.exclusivite_de_reprisereparation as exclusivite_de_reprisereparation_i,
	f.uniquement_sur_rdv as uniquement_sur_rdv_i,
	f.consignes_dacces as consignes_dacces_i,
	f.action_principale_id as action_principale_id_i,
	f.lieu_prestation as lieu_prestation_i,
	f.latitude as latitude_i,
	f.longitude as longitude_i,
	f.code_commune_insee as code_commune_insee_i,
	f.epci_id as epci_id_i,
	f.action_reparer as action_reparer_i,
	f.action_acheter as action_acheter_i,
	f.action_revendre as action_revendre_i,
	f.action_donner as action_donner_i,
	f.action_louer as action_louer_i,
	f.action_mettreenlocation as action_mettreenlocation_i,
	f.action_emprunter as action_emprunter_i,
	f.action_preter as action_preter_i,
	f.action_echanger as action_echanger_i,
	f.action_trier as action_trier_i,
	f.action_rapporter as action_rapporter_i,
	f2.nom as nom_j,
	f2.description as description_j,
	f2.acteur_type_id as acteur_type_id_j,
	f2.adresse as adresse_j,
	f2.adresse_complement as adresse_complement_j,
	f2.code_postal as code_postal_j,
	f2.ville as ville_j,
	f2.url as url_j,
	f2.email as email_j,
	f2.telephone as telephone_j,
	f2.nom_commercial as nom_commercial_j,
	f2.nom_officiel as nom_officiel_j,
	f2.siren as siren_j,
	f2.siret as siret_j,
	f2.source_id as source_id_j,
	f2.naf_principal as naf_principal_j,
	f2.horaires_osm as horaires_osm_j,
	f2.horaires_description as horaires_description_j,
	f2.public_accueilli as public_accueilli_j,
	f2.reprise as reprise_j,
	f2.exclusivite_de_reprisereparation as exclusivite_de_reprisereparation_j,
	f2.uniquement_sur_rdv as uniquement_sur_rdv_j,
	f2.consignes_dacces as consignes_dacces_j,
	f2.action_principale_id as action_principale_id_j,
	f2.lieu_prestation as lieu_prestation_j,
	f2.latitude as latitude_j,
	f2.longitude as longitude_j,
	f2.code_commune_insee as code_commune_insee_j,
	f2.epci_id as epci_id_j,
	f2.action_reparer as action_reparer_j,
	f2.action_acheter as action_acheter_j,
	f2.action_revendre as action_revendre_j,
	f2.action_donner as action_donner_j,
	f2.action_louer as action_louer_j,
	f2.action_mettreenlocation as action_mettreenlocation_j,
	f2.action_emprunter as action_emprunter_j,
	f2.action_preter as action_preter_j,
	f2.action_echanger as action_echanger_j,
	f2.action_trier as action_trier_j,
	f2.action_rapporter as action_rapporter_j
from
	luis.dataset_tmp dt
left join features f on
	dt.identifiant_unique_i = f.identifiant_unique
left join features f2 on
	dt.identifiant_unique_j = f2.identifiant_unique
"""

In [ ]:
df_features = pl.read_database_uri(sql_dataset_features, uri=DATABASE_URI)

In [ ]:
df_features.shape

In [ ]:
# Assert no duplicates
assert df_features.group_by(
    ["identifiant_unique_i", "identifiant_unique_j"]
).len().select(pl.col("len")).sum().item() == len(df_features)

In [ ]:
df_features.describe()

# Traitement des "__empty__"

In [38]:
df_features_preprocessed = df_features.with_columns(
    pl.selectors.by_dtype(pl.String)
    .str.strip_chars()
    .replace("__empty__", None)
    .replace("", None)
)

In [39]:
df_features_preprocessed.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2000""","""2000""",2000.0,"""2000""","""72""",2000.0,"""1975""","""177""","""1985""","""1998""","""303""","""175""","""558""","""554""","""9""","""1013""","""875""",1600.0,"""152""","""8""","""47""","""1521""","""329""",331.0,378.0,"""258""",14.0,"""574""",1998.0,1998.0,"""1951""",1950.0,1971.0,1971.0,1971.0,1971.0,…,"""1947""","""120""","""1987""","""1986""","""273""","""224""","""554""","""497""","""7""","""919""","""917""",1927.0,"""271""","""6""","""60""","""1405""","""232""",334.0,328.0,"""281""",12.0,"""466""",1996.0,1996.0,"""1938""",1937.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0
"""null_count""","""0""","""0""",0.0,"""0""","""1928""",0.0,"""25""","""1823""","""15""","""2""","""1697""","""1825""","""1442""","""1446""","""1991""","""987""","""1125""",400.0,"""1848""","""1992""","""1953""","""479""","""1671""",1669.0,1622.0,"""1742""",1986.0,"""1426""",2.0,2.0,"""49""",50.0,29.0,29.0,29.0,29.0,…,"""53""","""1880""","""13""","""14""","""1727""","""1776""","""1446""","""1503""","""1993""","""1081""","""1083""",73.0,"""1729""","""1994""","""1940""","""595""","""1768""",1666.0,1672.0,"""1719""",1988.0,"""1534""",4.0,4.0,"""62""",63.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
"""mean""",null,null,0.5,null,null,6.027,null,null,null,null,null,null,null,null,null,null,null,112.75,null,null,null,null,null,0.0,0.002646,null,10.5,null,2.375004,45.733278,null,570.378974,0.109082,0.023338,0.008118,0.069508,…,null,null,null,null,null,null,null,null,null,null,null,120.618578,null,null,null,null,null,0.002994,0.02439,null,10.416667,null,2.377214,45.808445,null,559.986577,0.172777,0.031642,0.006027,0.066298,0.008036,0.0,0.020593,0.0,0.008036,0.767454,0.010045
"""std""",null,null,null,null,null,2.847336,null,null,null,null,null,null,null,null,null,null,null,105.789217,null,null,null,null,null,null,null,null,1.870829,null,7.170461,5.747068,null,354.747249,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,108.478074,null,null,null,null,null,null,null,null,2.020726,null,7.029302,5.522328,null,349.470086,null,null,null,null,null,null,null,null,null,null,null
"""min""","""016256a6-f4be-5bf6-867a-487a54…","""1e0bfede-8c6b-5469-9b92-b458bf…",0.0,"""2 A Sports""","""Atelier bois + ressourcerie""",1.0,"""/""","""(sortie Carrefour)""","""01000""","""ALLEX""","""http://raid-info.fr/""","""BAYONNE.sav@leroymerlin.fr""","""+33240963172""","""ALDIS""","""AMBIANCES DU PASSE""","""054800958""","""05480095800313""",1.0,"""13.92Z""","""Mo 10:00-13:00,14:30-19:00;

# Traitement de l'absence de latitude/longitude

In [40]:
df_features_preprocessed = df_features_preprocessed.with_columns(
    pl.selectors.contains("latitude","longitude")
    .fill_null(0)
)

In [41]:
df_features_preprocessed.describe()

statistic,identifiant_unique_i,identifiant_unique_j,label,nom_i,description_i,acteur_type_id_i,adresse_i,adresse_complement_i,code_postal_i,ville_i,url_i,email_i,telephone_i,nom_commercial_i,nom_officiel_i,siren_i,siret_i,source_id_i,naf_principal_i,horaires_osm_i,horaires_description_i,public_accueilli_i,reprise_i,exclusivite_de_reprisereparation_i,uniquement_sur_rdv_i,consignes_dacces_i,action_principale_id_i,lieu_prestation_i,latitude_i,longitude_i,code_commune_insee_i,epci_id_i,action_reparer_i,action_acheter_i,action_revendre_i,action_donner_i,…,adresse_j,adresse_complement_j,code_postal_j,ville_j,url_j,email_j,telephone_j,nom_commercial_j,nom_officiel_j,siren_j,siret_j,source_id_j,naf_principal_j,horaires_osm_j,horaires_description_j,public_accueilli_j,reprise_j,exclusivite_de_reprisereparation_j,uniquement_sur_rdv_j,consignes_dacces_j,action_principale_id_j,lieu_prestation_j,latitude_j,longitude_j,code_commune_insee_j,epci_id_j,action_reparer_j,action_acheter_j,action_revendre_j,action_donner_j,action_louer_j,action_mettreenlocation_j,action_emprunter_j,action_preter_j,action_echanger_j,action_trier_j,action_rapporter_j
str,str,str,f64,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,…,str,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,f64,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""2000""","""2000""",2000.0,"""2000""","""72""",2000.0,"""1975""","""177""","""1985""","""1998""","""303""","""175""","""558""","""554""","""9""","""1013""","""875""",1600.0,"""152""","""8""","""47""","""1521""","""329""",331.0,378.0,"""258""",14.0,"""574""",2000.0,2000.0,"""1951""",1950.0,1971.0,1971.0,1971.0,1971.0,…,"""1947""","""120""","""1987""","""1986""","""273""","""224""","""554""","""497""","""7""","""919""","""917""",1927.0,"""271""","""6""","""60""","""1405""","""232""",334.0,328.0,"""281""",12.0,"""466""",2000.0,2000.0,"""1938""",1937.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0,1991.0
"""null_count""","""0""","""0""",0.0,"""0""","""1928""",0.0,"""25""","""1823""","""15""","""2""","""1697""","""1825""","""1442""","""1446""","""1991""","""987""","""1125""",400.0,"""1848""","""1992""","""1953""","""479""","""1671""",1669.0,1622.0,"""1742""",1986.0,"""1426""",0.0,0.0,"""49""",50.0,29.0,29.0,29.0,29.0,…,"""53""","""1880""","""13""","""14""","""1727""","""1776""","""1446""","""1503""","""1993""","""1081""","""1083""",73.0,"""1729""","""1994""","""1940""","""595""","""1768""",1666.0,1672.0,"""1719""",1988.0,"""1534""",0.0,0.0,"""62""",63.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0
"""mean""",null,null,0.5,null,null,6.027,null,null,null,null,null,null,null,null,null,null,null,112.75,null,null,null,null,null,0.0,0.002646,null,10.5,null,2.372629,45.687545,null,570.378974,0.109082,0.023338,0.008118,0.069508,…,null,null,null,null,null,null,null,null,null,null,null,120.618578,null,null,null,null,null,0.002994,0.02439,null,10.416667,null,2.372459,45.716828,null,559.986577,0.172777,0.031642,0.006027,0.066298,0.008036,0.0,0.020593,0.0,0.008036,0.767454,0.010045
"""std""",null,null,null,null,null,2.847336,null,null,null,null,null,null,null,null,null,null,null,105.789217,null,null,null,null,null,null,null,null,1.870829,null,7.167267,5.923363,null,354.747249,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,108.478074,null,null,null,null,null,null,null,null,2.020726,null,7.023069,5.884353,null,349.470086,null,null,null,null,null,null,null,null,null,null,null
"""min""","""016256a6-f4be-5bf6-867a-487a54…","""1e0bfede-8c6b-5469-9b92-b458bf…",0.0,"""2 A Sports""","""Atelier bois + ressourcerie""",1.0,"""/""","""(sortie Carrefour)""","""01000""","""ALLEX""","""http://raid-info.fr/""","""BAYONNE.sav@leroymerlin.fr""","""+33240963172""","""ALDIS""","""AMBIANCES DU PASSE""","""054800958""","""05480095800313""",1.0,"""13.92Z""","""Mo 10:00-13:00,14:30-19:00;

# Train/test split

In [42]:
df_train, df_test = train_test_split(df_features_preprocessed, test_size=0.2,random_state=RANDOM_SEED)

In [43]:
df_train.group_by("label").len()

label,len
bool,u32
true,808
false,792


In [44]:
df_test.group_by("label").len()

label,len
bool,u32
true,192
false,208


In [45]:
id_cols = ["identifiant_unique_i", "identifiant_unique_j"]
colnames = [
    "identifiant_unique_i",
    "identifiant_unique_j",
    "label",
    "nom_i",
    "description_i",
    "acteur_type_id_i",
    "adresse_i",
    "adresse_complement_i",
    "code_postal_i",
    "ville_i",
    "url_i",
    "email_i",
    "telephone_i",
    "nom_commercial_i",
    "nom_officiel_i",
    "siren_i",
    "siret_i",
    "source_id_i",
    "naf_principal_i",
    "horaires_osm_i",
    "horaires_description_i",
    "public_accueilli_i",
    "reprise_i",
    "exclusivite_de_reprisereparation_i",
    "uniquement_sur_rdv_i",
    "consignes_dacces_i",
    "action_principale_id_i",
    "lieu_prestation_i",
    "latitude_i",
    "longitude_i",
    "code_commune_insee_i",
    "epci_id_i",
    "action_reparer_i",
    "action_acheter_i",
    "action_revendre_i",
    "action_donner_i",
    "action_louer_i",
    "action_mettreenlocation_i",
    "action_emprunter_i",
    "action_preter_i",
    "action_echanger_i",
    "action_trier_i",
    "action_rapporter_i",
    "nom_j",
    "description_j",
    "acteur_type_id_j",
    "adresse_j",
    "adresse_complement_j",
    "code_postal_j",
    "ville_j",
    "url_j",
    "email_j",
    "telephone_j",
    "nom_commercial_j",
    "nom_officiel_j",
    "siren_j",
    "siret_j",
    "source_id_j",
    "naf_principal_j",
    "horaires_osm_j",
    "horaires_description_j",
    "public_accueilli_j",
    "reprise_j",
    "exclusivite_de_reprisereparation_j",
    "uniquement_sur_rdv_j",
    "consignes_dacces_j",
    "action_principale_id_j",
    "lieu_prestation_j",
    "latitude_j",
    "longitude_j",
    "code_commune_insee_j",
    "epci_id_j",
    "action_reparer_j",
    "action_acheter_j",
    "action_revendre_j",
    "action_donner_j",
    "action_louer_j",
    "action_mettreenlocation_j",
    "action_emprunter_j",
    "action_preter_j",
    "action_echanger_j",
    "action_trier_j",
    "action_rapporter_j",
]

In [46]:
df_features_preprocessed_split = (
    df_features_preprocessed.join(
        df_train.select(*id_cols, "label"),
        on=id_cols,
        how="left",
        validate="1:1",
        suffix="_train",
    )
    .join(
        df_test.select(*id_cols, "label"),
        on=id_cols,
        how="left",
        validate="1:1",
        suffix="_test",
    )
    .with_columns(
        pl.when(pl.col("label_train").is_not_null())
        .then(pl.lit("train"))
        .when(pl.col("label_test").is_not_null())
        .then(pl.lit("test"))
        .otherwise(pl.lit("unknown"))
        .alias("split")
    )
    .select(colnames + ["split"])
)

# Sauvegarde

In [47]:
df_features_preprocessed_split.write_parquet(
    DATASET_PATH.parent / f"features_{DATASET_PATH.stem}.parquet"
)